In [ ]:
# --- Repository setup ---
import os
from pathlib import Path

# Find the repository root (the folder that contains data/graphs) and work from there,
# so every relative path below resolves no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'graphs').is_dir())
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)


# Build evaluation table from JSON (two-LLM baseline) runs

This notebook scans `results/raw_runs/S2_llm_graph_json/**` for per-run JSON files, extracts evaluation counts, model specs, and token usage, displays a table, and exports a CSV.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

ROOT = Path('results/raw_runs/S2_llm_graph_json')
if not ROOT.exists():
    raise FileNotFoundError(f'JSON eval root not found: {ROOT.resolve()}')

print('Scanning:', ROOT.resolve())

Scanning: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase2_eval_runs/json


In [3]:
# Find all JSON files under ROOT
json_files = sorted(p for p in ROOT.rglob('*.json') if p.is_file())
print('Found JSON files:', len(json_files))

# Preview a few
for p in json_files[:5]:
    print('-', p.relative_to(ROOT))

Found JSON files: 132
- CMH/CMH_single_hop_single_hop_negative_1__CMH_single_hop_negative_1__base__nav-gemini-gemini-2.5-flash.json
- CMH/CMH_single_hop_single_hop_negative_1__CMH_single_hop_negative_1__base__nav-gemini-gemini-2.5-pro.json
- CMH/CMH_single_hop_single_hop_negative_1__CMH_single_hop_negative_1__base__nav-gemini-gemini-3.5-flash.json
- CMH/CMH_single_hop_single_hop_negative_1__CMH_single_hop_negative_1__base__nav-openai-gpt-4.1.json
- CMH/CMH_single_hop_single_hop_negative_1__CMH_single_hop_negative_1__base__nav-openai-gpt-4o-mini.json


In [4]:
def safe_int(x: Any) -> int | None:
    if x is None:
        return None
    try:
        return int(x)
    except Exception:
        return None


def safe_str(x: Any) -> str | None:
    if x is None:
        return None
    try:
        s = str(x)
        return s if s else None
    except Exception:
        return None


def extract_nav_model_spec(data: dict[str, Any]) -> dict[str, Any]:
    # Prefer top-level nav_model_spec, fallback to meta.nav_model_spec
    spec = data.get('nav_model_spec')
    if not isinstance(spec, dict):
        spec = (data.get('meta') or {}).get('nav_model_spec')
    if not isinstance(spec, dict):
        spec = {}
    return {
        'nav_provider': safe_str(spec.get('provider')),
        'nav_model': safe_str(spec.get('model')),
    }


def extract_row(path: Path) -> dict[str, Any]:
    data = json.loads(path.read_text(encoding='utf-8'))

    # Module folder is the first directory under ROOT (e.g., DEP, SUI)
    rel_parts = path.relative_to(ROOT).parts
    module_folder = rel_parts[0] if len(rel_parts) >= 2 else path.parent.name

    evaluation = data.get('evaluation') or {}
    counts = evaluation.get('counts') or {}

    usage = data.get('usage') or {}
    vp = usage.get('vp') or {}
    nav = usage.get('nav') or {}

    row = {
        'file_name': path.name,
        'module_folder': module_folder,
        'case_id': safe_str(data.get('case_id')),
        'expected_assessment_findings': safe_int(counts.get('expected_assessment_findings')),
        'matched': safe_int(counts.get('matched')),
        'missing': safe_int(counts.get('missing')),
        'mismatched': safe_int(counts.get('mismatched')),
        'extra': safe_int(counts.get('extra')),
        # VP usage
        'vp_input_tokens': safe_int(vp.get('input_tokens')),
        'vp_output_tokens': safe_int(vp.get('output_tokens')),
        'vp_total_tokens': safe_int(vp.get('total_tokens')),
        # NAV usage
        'nav_calls': safe_int(nav.get('calls')),
        'nav_input_tokens': safe_int(nav.get('input_tokens')),
        'nav_output_tokens': safe_int(nav.get('output_tokens')),
        'nav_total_tokens': safe_int(nav.get('total_tokens')),
    }

    row.update(extract_nav_model_spec(data))
    return row


rows = [extract_row(p) for p in json_files]
print('Rows:', len(rows))

Rows: 132


In [5]:
# Display as a table
try:
    import pandas as pd

    df = pd.DataFrame(rows)
    df = df.sort_values(['module_folder', 'case_id', 'file_name']).reset_index(drop=True)
    df
except Exception as e:
    print('pandas not available; showing first 10 rows as dicts. Error:', str(e))
    for r in rows[:10]:
        print(r)

In [6]:
# Export CSV
OUTPUT_CSV = Path('results/summaries/S2_llm_graph_json_summary.csv')

columns = [
    'case_id',
    'module_folder',
    'expected_assessment_findings',
    'matched',
    'missing',
    'mismatched',
    'extra',
    'nav_provider',
    'nav_model',
    'vp_input_tokens',
    'vp_output_tokens',
    'vp_total_tokens',
    'nav_calls',
    'nav_input_tokens',
    'nav_output_tokens',
    'nav_total_tokens',
    'file_name',
]

try:
    import pandas as pd

    out_df = pd.DataFrame(rows)[columns].sort_values(['module_folder', 'case_id', 'nav_provider', 'nav_model', 'file_name'])
    out_df.to_csv(OUTPUT_CSV, index=False)
    print('Saved CSV:', OUTPUT_CSV.resolve())
except Exception:
    import csv

    with OUTPUT_CSV.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=columns)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k) for k in columns})
    print('Saved CSV (csv module):', OUTPUT_CSV.resolve())

Saved CSV: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase2_eval_runs/json_eval_summary.csv


In [7]:
out_df

,case_id,module_folder,expected_assessment_findings,matched,missing,mismatched,extra,nav_provider,nav_model,vp_input_tokens,vp_output_tokens,vp_total_tokens,nav_calls,nav_input_tokens,nav_output_tokens,nav_total_tokens,file_name
0,CMH_single_hop_negative_1,CMH,11,11,0,0,0,gemini,gemini-2.5-flash,7244,22,7266,11,52412,216,54537,CMH_single_hop_single_hop_negative_1__CMH_sing...
1,CMH_single_hop_negative_1,CMH,11,11,0,0,0,gemini,gemini-2.5-pro,7244,22,7266,11,52412,255,58844,CMH_single_hop_single_hop_negative_1__CMH_sing...
2,CMH_single_hop_negative_1,CMH,11,11,0,0,0,gemini,gemini-3.5-flash,7244,22,7266,11,52412,161,55075,CMH_single_hop_single_hop_negative_1__CMH_sing...
3,CMH_single_hop_negative_1,CMH,11,11,0,0,0,openai,gpt-4.1,7244,22,7266,11,49853,132,49985,CMH_single_hop_single_hop_negative_1__CMH_sing...
5,CMH_single_hop_negative_1,CMH,11,11,0,0,0,openai,gpt-4o,7244,22,7266,11,49853,179,50032,CMH_single_hop_single_hop_negative_1__CMH_sing...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,multi_hop_positive_SUI_to_OTH_1,SUI,11,11,0,0,0,gemini,gemini-2.5-pro,5694,22,5716,15,29702,348,40236,SUI_to_OTH_multi_hop_multi_hop_positive_1__mul...
128,multi_hop_positive_SUI_to_OTH_1,SUI,11,11,0,0,0,gemini,gemini-3.5-flash,5694,22,5716,15,29702,208,33628,SUI_to_OTH_multi_hop_multi_hop_positive_1__mul...
129,multi_hop_positive_SUI_to_OTH_1,SUI,11,11,0,0,0,openai,gpt-4.1,5694,22,5716,15,29135,173,29308,SUI_to_OTH_multi_hop_multi_hop_positive_1__mul...
131,multi_hop_positive_SUI_to_OTH_1,SUI,11,11,0,0,0,openai,gpt-4o,5694,22,5716,15,29135,236,29371,SUI_to_OTH_multi_hop_multi_hop_positive_1__mul...


In [ ]:
# (Optional) module-level summary (sums token counts and counts fields)
try:
    import pandas as pd

    df = pd.DataFrame(rows)
    summary = (
        df.groupby(['module_folder', 'nav_provider', 'nav_model'], dropna=False)
        .agg(
            files=('file_name', 'count'),
            expected=('expected_assessment_findings', 'sum'),
            matched=('matched', 'sum'),
            missing=('missing', 'sum'),
            mismatched=('mismatched', 'sum'),
            extra=('extra', 'sum'),
            vp_input_tokens=('vp_input_tokens', 'sum'),
            vp_output_tokens=('vp_output_tokens', 'sum'),
            vp_total_tokens=('vp_total_tokens', 'sum'),
            nav_calls=('nav_calls', 'sum'),
            nav_input_tokens=('nav_input_tokens', 'sum'),
            nav_output_tokens=('nav_output_tokens', 'sum'),
            nav_total_tokens=('nav_total_tokens', 'sum'),
        )
        .reset_index()
        .sort_values(['module_folder', 'nav_provider', 'nav_model'])
    )
    summary
except Exception:
    print('Skipping summary (pandas not available).')